# Churn Prediction in Non-Contractual Relationships Based on RFM Analysis

**AUTHOR**  
Rodrigo Kang

## Overview

This notebook accompanies the project **Customer Value and Retention** and develops the churn prediction component of the retail analytics workflow.

The previous notebook, **Customer Segmentation with RFM Analysis**, built a customer-level segmentation model using Recency, Frequency, and Monetary Value features from AdventureWorks transactions. This notebook uses the same analytical philosophy, but changes the objective from describing customer structure to estimating churn risk in a non-contractual setting.

Because there is no explicit cancellation event in retail transactions, churn is treated as a behavioural outcome. The observation window is split into a calibration period and a validation period. Customer features are built only from the calibration period, while the churn outcome is observed in the validation period. This prevents future purchases from leaking into the predictors and makes the modelling setup closer to a practical retention problem.

The modelling stage benchmarks three classifiers: Logistic Regression, Random Forest, and XGBoost. Each model is evaluated under three sampling strategies: the original class distribution, random undersampling, and SMOTE. The best model is selected using validation performance, with emphasis on discriminative ability and practical usefulness for identifying customers at risk.

## Libraries and Configuration

The notebook uses the same core libraries as the RFM segmentation workflow, with additional tools for supervised learning, class imbalance handling, and model evaluation.

The implementation is intentionally explicit rather than overly abstract. This keeps the modelling decisions visible and makes the workflow easier to review, adapt, and reproduce.

In [1]:
# Data manipulation
import numpy as np
import pandas as pd

# Database connectivity
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Visualization
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Statistical transformations
from scipy.stats import boxcox

# Preprocessing and modelling
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    roc_curve,
    precision_recall_curve
)

# Class imbalance strategies
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# Miscellaneous
from pathlib import Path
import warnings

In [2]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

pio.renderers.default = "notebook_connected"

seed = 42
np.random.seed(seed)

outputs_dir = Path("1-customer-value-and-retention-churn-figures")
tables_dir = outputs_dir / "tables"

outputs_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)


def clean_filename(name):
    """
    Convert a table or figure name into a simple file-friendly stem.

    Inputs:
    -------
        name : str
            Original table or figure name.

    Outputs:
    --------
        str
            Sanitized filename.

    Author:
    -------
        Rodrigo Kang
    """

    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "-")
        .replace("_", "-")
    )


def save_table(data, filename, index=False):
    """
    Save a dataframe as CSV in the churn output tables folder.

    Inputs:
    -------
        data : pandas.DataFrame
            Table to export.

        filename : str
            Output filename without extension.

        index : bool, default=False
            Whether to save the dataframe index.

    Outputs:
    --------
        pathlib.Path
            Path to the exported CSV file.

    Author:
    -------
        Rodrigo Kang
    """

    output_path = tables_dir / f"{clean_filename(filename)}.csv"
    data.to_csv(output_path, index=index)

    return output_path


def save_plotly_figure(fig, filename, width=1000, height=650, scale=2):
    """
    Save a Plotly figure as interactive HTML and static PNG.

    Notes:
    ------
        PNG export requires the kaleido package.

    Inputs:
    -------
        fig : plotly.graph_objects.Figure
            Figure to export.

        filename : str
            Output filename without extension.

        width : int, default=1000
            Figure width in pixels.

        height : int, default=650
            Figure height in pixels.

        scale : int, default=2
            Resolution scaling factor for PNG export.

    Outputs:
    --------
        tuple[pathlib.Path, pathlib.Path]
            Paths to the exported HTML and PNG files.

    Author:
    -------
        Rodrigo Kang
    """

    stem = clean_filename(filename)

    html_path = outputs_dir / f"{stem}.html"
    png_path = outputs_dir / f"{stem}.png"

    fig.write_html(html_path)

    try:
        fig.write_image(
            png_path,
            width=width,
            height=height,
            scale=scale
        )
    except Exception as error:
        print(
            "PNG export failed. Install or update kaleido if needed: "
            f"{error}"
        )

    return html_path, png_path


def apply_plotly_layout(fig, title, xaxis_title=None, yaxis_title=None):
    """
    Apply a consistent visual layout to portfolio figures.

    Inputs:
    -------
        fig : plotly.graph_objects.Figure
            Figure to format.

        title : str
            Figure title.

        xaxis_title : str, optional
            Label for the x-axis.

        yaxis_title : str, optional
            Label for the y-axis.

    Outputs:
    --------
        plotly.graph_objects.Figure
            Formatted Plotly figure.

    Author:
    -------
        Rodrigo Kang
    """

    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1000,
        height=650,
        font=dict(
            family="Arial",
            size=14
        ),
        title_font=dict(size=20),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=70, r=40, t=90, b=70)
    )

    if xaxis_title is not None:
        fig.update_xaxes(title_text=xaxis_title)

    if yaxis_title is not None:
        fig.update_yaxes(title_text=yaxis_title)

    return fig

## Database Connection

The AdventureWorks database is hosted locally in PostgreSQL. Credentials are loaded from a local configuration file, avoiding the need to expose them inside the notebook.

A short validation query is executed after creating the connection to confirm that the database is available.

In [3]:
from local_config import DB_USER, DB_PASSWORD

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host="localhost",
    port=5432,
    database="adventureworks",
)

engine = create_engine(connection_url)

In [4]:
query = """
SELECT
    current_database() AS database_name,
    current_user AS user_name,
    current_setting('server_encoding') AS server_encoding,
    current_setting('client_encoding') AS client_encoding;
"""

pd.read_sql(query, engine)

,database_name,user_name,server_encoding,client_encoding
0,adventureworks,postgres,UTF8,UTF8


## Data Extraction

The extraction query follows the same customer analytics data model used in the RFM segmentation notebook. The dataset is kept at order-line level first, because this preserves flexibility for customer-level aggregation.

The fields used later include customer identifiers, order dates, sales order identifiers, product information, territory information, and line-level revenue.

In [5]:
query = """
SELECT
    c.customerid AS customer_id,
    soh.salesorderid AS sales_order_id,
    soh.orderdate AS order_date,
    soh.duedate AS due_date,
    soh.shipdate AS ship_date,
    soh.status AS order_status,
    soh.territoryid AS territory_id,
    st.name AS territory_name,
    st.countryregioncode AS country_region_code,
    sod.salesorderdetailid AS sales_order_detail_id,
    sod.productid AS product_id,
    p.name AS product_name,
    pc.name AS product_category,
    ps.name AS product_subcategory,
    sod.orderqty AS order_quantity,
    sod.unitprice AS unit_price,
    sod.unitpricediscount AS unit_price_discount,
    (
        sod.orderqty
        * sod.unitprice
        * (1 - sod.unitpricediscount)
    ) AS line_total
FROM sales.customer AS c
INNER JOIN sales.salesorderheader AS soh
    ON c.customerid = soh.customerid
INNER JOIN sales.salesorderdetail AS sod
    ON soh.salesorderid = sod.salesorderid
INNER JOIN production.product AS p
    ON sod.productid = p.productid
LEFT JOIN production.productsubcategory AS ps
    ON p.productsubcategoryid = ps.productsubcategoryid
LEFT JOIN production.productcategory AS pc
    ON ps.productcategoryid = pc.productcategoryid
LEFT JOIN sales.salesterritory AS st
    ON soh.territoryid = st.territoryid;
"""

transactions = pd.read_sql(query, engine)

transactions["order_date"] = pd.to_datetime(transactions["order_date"])
transactions["line_total"] = transactions["line_total"].astype(float)

transactions.head()

,customer_id,sales_order_id,order_date,due_date,ship_date,order_status,territory_id,territory_name,country_region_code,sales_order_detail_id,product_id,product_name,product_category,product_subcategory,order_quantity,unit_price,unit_price_discount,line_total
0,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,1,776,"Mountain-100 Black, 42",Bikes,Mountain Bikes,1,2024.994,0.0,2024.994
1,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,2,777,"Mountain-100 Black, 44",Bikes,Mountain Bikes,3,2024.994,0.0,6074.982
2,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,3,778,"Mountain-100 Black, 48",Bikes,Mountain Bikes,1,2024.994,0.0,2024.994
3,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,4,771,"Mountain-100 Silver, 38",Bikes,Mountain Bikes,1,2039.994,0.0,2039.994
4,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,5,772,"Mountain-100 Silver, 42",Bikes,Mountain Bikes,1,2039.994,0.0,2039.994


In [6]:
inspection_summary = pd.DataFrame({
    "metric": [
        "Order lines",
        "Customers",
        "Orders",
        "Products",
        "Minimum order date",
        "Maximum order date",
        "Total revenue"
    ],
    "value": [
        transactions.shape[0],
        transactions["customer_id"].nunique(),
        transactions["sales_order_id"].nunique(),
        transactions["product_id"].nunique(),
        transactions["order_date"].min(),
        transactions["order_date"].max(),
        round(transactions["line_total"].sum(), 2)
    ]
})

save_table(inspection_summary, "inspection-summary")

inspection_summary

,metric,value
0,Order lines,121317
1,Customers,19119
2,Orders,31465
3,Products,266
4,Minimum order date,2022-05-30 00:00:00
5,Maximum order date,2025-06-29 00:00:00
6,Total revenue,109846381.4


## Calibration and Validation Windows

The full transaction history runs from **2022-05-30** to **2025-06-29**. To turn churn into a predictive problem, the window is split as follows:

- **Calibration period:** transactions before the final year.
- **Validation period:** the final year of observed transactions.

RFM features, latency, and customer segment assignments are computed using only the calibration period. The validation period is then used to observe whether customers returned after the calibration cut-off.

In [7]:
min_order_date = transactions["order_date"].min().normalize()
max_order_date = transactions["order_date"].max().normalize()

validation_start = max_order_date - pd.DateOffset(years=1) + pd.Timedelta(days=1)
validation_end = max_order_date

calibration_start = min_order_date
calibration_end = validation_start - pd.Timedelta(days=1)

window_summary = pd.DataFrame({
    "window": ["Calibration", "Validation"],
    "start_date": [calibration_start, validation_start],
    "end_date": [calibration_end, validation_end]
})

save_table(window_summary, "window-summary")

window_summary

,window,start_date,end_date
0,Calibration,2022-05-30,2024-06-29
1,Validation,2024-06-30,2025-06-29


In [8]:
calibration_transactions = transactions[
    (transactions["order_date"] >= calibration_start)
    & (transactions["order_date"] <= calibration_end)
].copy()

validation_transactions = transactions[
    (transactions["order_date"] >= validation_start)
    & (transactions["order_date"] <= validation_end)
].copy()

period_summary = pd.DataFrame({
    "period": ["Calibration", "Validation"],
    "order_lines": [
        calibration_transactions.shape[0],
        validation_transactions.shape[0]
    ],
    "customers": [
        calibration_transactions["customer_id"].nunique(),
        validation_transactions["customer_id"].nunique()
    ],
    "orders": [
        calibration_transactions["sales_order_id"].nunique(),
        validation_transactions["sales_order_id"].nunique()
    ],
    "revenue": [
        calibration_transactions["line_total"].sum(),
        validation_transactions["line_total"].sum()
    ]
})

save_table(period_summary.round(2), "period-summary")

period_summary.round(2)

,period,order_lines,customers,orders,revenue
0,Calibration,43033,6206,8263,64841795.92
1,Validation,78284,18069,23202,45004585.48


## Calibration RFM Features

The RFM construction follows the segmentation notebook, with one important change: the reference date is now the first day of the validation window.

This means that recency is measured as the number of days between the customer’s last calibration purchase and the moment at which a retention decision would be made.

In [9]:
calibration_reference_date = validation_start

rfm = (
    calibration_transactions
    .groupby("customer_id")
    .agg(
        recency=(
            "order_date",
            lambda x: (calibration_reference_date - x.max()).days
        ),
        frequency=("sales_order_id", "nunique"),
        monetary=("line_total", "sum"),
        first_purchase_date=("order_date", "min"),
        last_purchase_date=("order_date", "max"),
        total_quantity=("order_quantity", "sum"),
        product_diversity=("product_id", "nunique"),
        category_diversity=("product_category", "nunique"),
        territory_name=("territory_name", "first")
    )
    .reset_index()
)

rfm.head()

,customer_id,recency,frequency,monetary,first_purchase_date,last_purchase_date,total_quantity,product_diversity,category_diversity,territory_name
0,11000,11,2,5741.96,2022-06-20,2024-06-19,3,3,2,Australia
1,11001,13,2,5794.92,2022-06-16,2024-06-17,7,7,3,Australia
2,11002,29,2,5694.98,2022-06-08,2024-06-01,2,2,1,Australia
3,11003,24,2,5718.95,2022-05-30,2024-06-06,5,5,3,Australia
4,11004,7,2,5776.95,2022-06-24,2024-06-23,4,4,2,Australia


In [10]:
def calculate_average_latency(data):
    """
    Calculate the average number of days between purchases for each customer.

    Inputs:
    -------
        data (pd.DataFrame): Calibration-period transactions with customer_id,
            sales_order_id, and order_date columns.

    Outputs:
    --------
        pd.DataFrame: Customer-level table with average_latency and
            observed_intervals.

    Author:
    -------
        Rodrigo Kang
    """

    order_dates = (
        data[["customer_id", "sales_order_id", "order_date"]]
        .drop_duplicates()
        .sort_values(["customer_id", "order_date"])
        .copy()
    )

    order_dates["latency_days"] = (
        order_dates
        .groupby("customer_id")["order_date"]
        .diff()
        .dt.days
    )

    latency = (
        order_dates
        .groupby("customer_id")
        .agg(
            average_latency=("latency_days", "mean"),
            observed_intervals=("latency_days", "count")
        )
        .reset_index()
    )

    return latency

latency = calculate_average_latency(calibration_transactions)

rfm = rfm.merge(
    latency,
    on="customer_id",
    how="left"
)

# Single-purchase customers have no observed interpurchase interval.
# Their current recency is used as a conservative latency proxy.
rfm["average_latency"] = rfm["average_latency"].fillna(rfm["recency"])
rfm["observed_intervals"] = rfm["observed_intervals"].fillna(0).astype(int)

rfm.head()

,customer_id,recency,frequency,monetary,first_purchase_date,last_purchase_date,total_quantity,product_diversity,category_diversity,territory_name,average_latency,observed_intervals
0,11000,11,2,5741.96,2022-06-20,2024-06-19,3,3,2,Australia,730.0,1
1,11001,13,2,5794.92,2022-06-16,2024-06-17,7,7,3,Australia,732.0,1
2,11002,29,2,5694.98,2022-06-08,2024-06-01,2,2,1,Australia,724.0,1
3,11003,24,2,5718.95,2022-05-30,2024-06-06,5,5,3,Australia,738.0,1
4,11004,7,2,5776.95,2022-06-24,2024-06-23,4,4,2,Australia,730.0,1


In [11]:
rfm_summary = rfm[
    ["recency", "frequency", "monetary", "average_latency"]
].describe().T

rfm_summary

,count,mean,std,min,25%,50%,75%,max
recency,6206.0,295.106832,221.339376,1.000,91.0000,263.0000,466.00,762.000000
frequency,6206.0,1.331453,1.182139,1.000,1.0000,1.0000,1.00,9.000000
monetary,6206.0,10448.242977,47361.564364,1.374,2049.0982,2181.5625,3578.27,640042.621126
average_latency,6206.0,314.686165,216.101407,0.000,113.1875,285.0000,490.75,762.000000


## RFM Transformation and Segmentation

The segmentation stage mirrors the previous notebook: Box-Cox transformations are applied to reduce skewness, the transformed variables are standardized, and K-Means is fitted using the three RFM scores.

The number of clusters is fixed at **K = 9**, consistent with the previous segmentation analysis.

In [12]:
def build_rfm_scores(data):
    """
    Transform RFM variables and create standardized RFM scores.

    Inputs:
    -------
        data (pd.DataFrame): Customer-level RFM table with recency,
            frequency, and monetary columns.

    Outputs:
    --------
        tuple:
            pd.DataFrame: Input table with Box-Cox variables and standardized
                score_recency, score_frequency, and score_monetary columns.
            dict: Fitted Box-Cox lambda values by RFM feature.
            StandardScaler: Fitted scaler used for the transformed RFM features.

    Author:
    -------
        Rodrigo Kang
    """

    output = data.copy()

    rfm_features = ["recency", "frequency", "monetary"]
    boxcox_lambdas = {}

    for feature in rfm_features:
        transformed_values, fitted_lambda = boxcox(output[feature] + 1)
        output[f"{feature}_boxcox"] = transformed_values
        boxcox_lambdas[feature] = fitted_lambda

    scaler = StandardScaler()

    output[[
        "score_recency",
        "score_frequency",
        "score_monetary"
    ]] = scaler.fit_transform(
        output[[
            "recency_boxcox",
            "frequency_boxcox",
            "monetary_boxcox"
        ]]
    )

    return output, boxcox_lambdas, scaler

rfm_transformed, boxcox_lambdas, rfm_scaler = build_rfm_scores(rfm)

boxcox_lambdas

{'recency': np.float64(0.4804767928363453),
 'frequency': np.float64(-10.488143723887374),
 'monetary': np.float64(-0.13804481116160153)}

In [13]:
k_final = 9

kmeans = KMeans(
    n_clusters=k_final,
    random_state=seed,
    n_init=20
)

clustering_features = [
    "score_recency",
    "score_frequency",
    "score_monetary"
]

rfm["cluster"] = kmeans.fit_predict(
    rfm_transformed[clustering_features]
)

rfm_transformed["cluster"] = rfm["cluster"]

In [14]:
cluster_profiles = (
    rfm
    .groupby("cluster")
    .agg(
        customers=("customer_id", "count"),
        recency=("recency", "mean"),
        frequency=("frequency", "mean"),
        monetary=("monetary", "mean"),
        average_latency=("average_latency", "mean")
    )
    .round(2)
    .sort_values("monetary", ascending=False)
)

cluster_profiles

,customers,recency,frequency,monetary,average_latency
cluster,,,,,
3,273,113.24,5.89,175385.89,93.02
7,53,10.94,1.00,43572.70,10.94
1,454,45.08,2.59,4886.17,324.94
0,1767,563.38,1.00,3543.65,563.38
4,928,80.88,1.00,2203.71,80.88
6,1395,277.42,1.00,2193.24,277.42
8,609,87.12,1.00,853.94,87.12
2,686,389.53,1.00,795.05,389.53
5,41,40.27,1.02,50.52,39.56


In [15]:
segment_names = {
    5: "Champions",
    1: "High Value but Cooling",
    7: "Lost Customers",
    8: "Recent Low-Value Customers",
    3: "Inactive Low-Value Customers",
    4: "Frequent Low-Spend Customers",
    6: "Declining Frequent Buyers",
    0: "Active Minimal Buyers",
    2: "Inactive Minimal Buyers"
}

segment_order = [
    "Champions",
    "High Value but Cooling",
    "Frequent Low-Spend Customers",
    "Recent Low-Value Customers",
    "Active Minimal Buyers",
    "Declining Frequent Buyers",
    "Inactive Low-Value Customers",
    "Inactive Minimal Buyers",
    "Lost Customers"
]

rfm["segment"] = rfm["cluster"].map(segment_names)

rfm[["customer_id", "recency", "frequency", "monetary", "cluster", "segment"]].head()

,customer_id,recency,frequency,monetary,cluster,segment
0,11000,11,2,5741.96,1,High Value but Cooling
1,11001,13,2,5794.92,1,High Value but Cooling
2,11002,29,2,5694.98,1,High Value but Cooling
3,11003,24,2,5718.95,1,High Value but Cooling
4,11004,7,2,5776.95,1,High Value but Cooling


The segment mapping is inherited from the full-period segmentation workflow. In a production setting, the labels should always be checked after refitting on a new time window, because K-Means cluster identifiers can change when the estimation sample changes.

In [16]:
def calculate_cluster_weights(data, cluster_column, score_columns):
    """
    Estimate intra-cluster RFM weights from the variance of standardized scores.

    Inputs:
    -------
        data (pd.DataFrame): Customer-level table containing cluster labels and
            standardized RFM score columns.
        cluster_column (str): Name of the column containing cluster labels.
        score_columns (list): Names of the standardized recency, frequency,
            and monetary score columns.

    Outputs:
    --------
        pd.DataFrame: Cluster-level weights that sum to one within each cluster.

    Author:
    -------
        Rodrigo Kang
    """

    cluster_variances = (
        data
        .groupby(cluster_column)[score_columns]
        .var()
    )

    cluster_weights = cluster_variances.div(
        cluster_variances.sum(axis=1),
        axis=0
    )

    return cluster_weights


def assign_rfm_score(data, cluster_column, score_columns, cluster_weights):
    """
    Assign a normalized intra-segment RFM score to each customer.

    Inputs:
        data (pd.DataFrame): Customer-level table containing RFM scores and
            cluster labels.
        cluster_column (str): Name of the column containing cluster labels.
        score_columns (list): Names of the standardized recency, frequency,
            and monetary score columns.
        cluster_weights (pd.DataFrame): Cluster-specific RFM weights produced by
            calculate_cluster_weights.

    Outputs:
        pd.DataFrame: Input table enriched with RFM weights, raw RFM score, and
            normalized RFM score.

    Author:
        Rodrigo Kang
    """

    output = data.copy()

    recency_col, frequency_col, monetary_col = score_columns

    output["omega_recency"] = output[cluster_column].map(
        cluster_weights[recency_col]
    )
    output["omega_frequency"] = output[cluster_column].map(
        cluster_weights[frequency_col]
    )
    output["omega_monetary"] = output[cluster_column].map(
        cluster_weights[monetary_col]
    )

    output["rfm_score"] = (
        - output[recency_col] * output["omega_recency"]
        + output[frequency_col] * output["omega_frequency"]
        + output[monetary_col] * output["omega_monetary"]
    )

    scaler = MinMaxScaler()
    output["rfm_normalized_score"] = scaler.fit_transform(
        output[["rfm_score"]]
    )

    return output

cluster_weights = calculate_cluster_weights(
    data=rfm_transformed,
    cluster_column="cluster",
    score_columns=clustering_features
)

scoring_data = (
    rfm
    .merge(
        rfm_transformed[
            [
                "customer_id",
                "score_recency",
                "score_frequency",
                "score_monetary"
            ]
        ],
        on="customer_id",
        how="left"
    )
)

rfm_scored = assign_rfm_score(
    data=scoring_data,
    cluster_column="cluster",
    score_columns=clustering_features,
    cluster_weights=cluster_weights
)

rfm_scored.head()

,customer_id,recency,frequency,monetary,first_purchase_date,last_purchase_date,total_quantity,product_diversity,category_diversity,territory_name,average_latency,observed_intervals,cluster,segment,score_recency,score_frequency,score_monetary,omega_recency,omega_frequency,omega_monetary,rfm_score,rfm_normalized_score
0,11000,11,2,5741.96,2022-06-20,2024-06-19,3,3,2,Australia,730.0,1,1,High Value but Cooling,-1.662684,2.719297,0.818697,0.519728,0.0006,0.479672,1.258481,0.868062
1,11001,13,2,5794.92,2022-06-16,2024-06-17,7,7,3,Australia,732.0,1,1,High Value but Cooling,-1.622934,2.719297,0.826561,0.519728,0.0006,0.479672,1.241594,0.866566
2,11002,29,2,5694.98,2022-06-08,2024-06-01,2,2,1,Australia,724.0,1,1,High Value but Cooling,-1.376703,2.719297,0.811652,0.519728,0.0006,0.479672,1.106469,0.854590
3,11003,24,2,5718.95,2022-05-30,2024-06-06,5,5,3,Australia,738.0,1,1,High Value but Cooling,-1.444056,2.719297,0.815255,0.519728,0.0006,0.479672,1.143202,0.857846
4,11004,7,2,5776.95,2022-06-24,2024-06-23,4,4,2,Australia,730.0,1,1,High Value but Cooling,-1.754208,2.719297,0.823902,0.519728,0.0006,0.479672,1.308545,0.872499


## Churn Definition

Customer churn is defined using a behavioural rule derived from customer purchasing activity during the calibration period. The objective is to identify customers who exhibit patterns consistent with disengagement and then verify whether they actually return during the validation period.

A customer is labelled as churned when all of the following conditions are met:

1. The customer belongs to a low-activity segment during the calibration window.
2. The customer's recency exceeds their average purchase latency.
3. The customer's RFM score is below the average score of their segment.
4. The customer makes no purchases during the validation window.

This definition combines customer value, purchase frequency, and inactivity patterns to identify customers at risk of attrition. By deriving the rule exclusively from information available during the calibration period and validating the outcome in a future observation window, the approach avoids information leakage while remaining consistent with the non-contractual nature of retail customer relationships.

In [17]:
validation_activity = (
    validation_transactions
    .groupby("customer_id")
    .agg(
        validation_orders=("sales_order_id", "nunique"),
        validation_revenue=("line_total", "sum"),
        validation_last_purchase=("order_date", "max")
    )
    .reset_index()
)

model_data = (
    rfm_scored
    .merge(
        validation_activity,
        on="customer_id",
        how="left"
    )
)

model_data["validation_orders"] = model_data["validation_orders"].fillna(0).astype(int)
model_data["validation_revenue"] = model_data["validation_revenue"].fillna(0.0)
model_data["returned_in_validation"] = (model_data["validation_orders"] > 0).astype(int)

In [18]:
low_activity_segments = [
    "Lost Customers",
    "Inactive Low-Value Customers",
    "Inactive Minimal Buyers",
    "Declining Frequent Buyers"
]

segment_average_scores = (
    model_data
    .groupby("segment")["rfm_normalized_score"]
    .mean()
)

model_data["segment_average_score"] = model_data["segment"].map(segment_average_scores)
model_data["delta_recency_latency"] = model_data["recency"] - model_data["average_latency"]

model_data["rfm_churn_rule"] = (
    model_data["segment"].isin(low_activity_segments)
    & (model_data["delta_recency_latency"] > 0)
    & (model_data["rfm_normalized_score"] < model_data["segment_average_score"])
).astype(int)

model_data["churn"] = (
    (model_data["rfm_churn_rule"] == 1)
    & (model_data["returned_in_validation"] == 0)
).astype(int)

model_data[[
    "customer_id",
    "segment",
    "recency",
    "average_latency",
    "delta_recency_latency",
    "rfm_normalized_score",
    "segment_average_score",
    "rfm_churn_rule",
    "returned_in_validation",
    "churn"
]].head()

,customer_id,segment,recency,average_latency,delta_recency_latency,rfm_normalized_score,segment_average_score,rfm_churn_rule,returned_in_validation,churn
0,11000,High Value but Cooling,11,730.0,-719.0,0.868062,0.842736,0,1,0
1,11001,High Value but Cooling,13,732.0,-719.0,0.866566,0.842736,0,1,0
2,11002,High Value but Cooling,29,724.0,-695.0,0.854590,0.842736,0,1,0
3,11003,High Value but Cooling,24,738.0,-714.0,0.857846,0.842736,0,1,0
4,11004,High Value but Cooling,7,730.0,-723.0,0.872499,0.842736,0,1,0


In [19]:
churn_summary = pd.DataFrame({
    "class": ["Retention", "Churn"],
    "customers": [
        (model_data["churn"] == 0).sum(),
        (model_data["churn"] == 1).sum()
    ]
})

churn_summary["percentage"] = (
    100 * churn_summary["customers"] / churn_summary["customers"].sum()
)

save_table(churn_summary, "churn-summary")

churn_summary

,class,customers,percentage
0,Retention,6141,98.952626
1,Churn,65,1.047374


In [20]:
segment_churn_summary = (
    model_data
    .groupby("segment")
    .agg(
        customers=("customer_id", "count"),
        churn_rate=("churn", "mean"),
        returned_rate=("returned_in_validation", "mean"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean")
    )
    .reset_index()
)

segment_churn_summary["segment"] = pd.Categorical(
    segment_churn_summary["segment"],
    categories=segment_order,
    ordered=True
)

segment_churn_summary = segment_churn_summary.sort_values("segment")
segment_churn_summary.round(4)

,segment,customers,churn_rate,returned_rate,avg_recency,avg_frequency,avg_monetary
1,Champions,41,0.0000,0.4878,40.2683,1.0244,50.5193
4,High Value but Cooling,454,0.0000,0.5088,45.0815,2.5859,4886.1746
3,Frequent Low-Spend Customers,928,0.0000,0.8006,80.8761,1.0000,2203.7077
8,Recent Low-Value Customers,609,0.0000,0.7947,87.1215,1.0000,853.9364
0,Active Minimal Buyers,1767,0.0000,0.8557,563.3786,1.0000,3543.6513
2,Declining Frequent Buyers,1395,0.0000,0.9154,277.4229,1.0000,2193.2446
5,Inactive Low-Value Customers,273,0.2381,0.6850,113.2381,5.8938,175385.8867
6,Inactive Minimal Buyers,686,0.0000,0.9461,389.5277,1.0000,795.0514
7,Lost Customers,53,0.0000,1.0000,10.9434,1.0000,43572.6981


## Feature Set

The modelling features are derived exclusively from customer behaviour observed during the calibration period. They include the original RFM variables, transformed RFM scores, the normalized RFM prioritisation score, purchase latency, segment information, and behavioural descriptors such as product and category diversity.

All predictors are computed using information available at the end of the calibration window. Data from the validation period is used only to determine the observed churn outcome.

In [21]:
numeric_features = [
    "recency",
    "frequency",
    "monetary",
    "average_latency",
    "observed_intervals",
    "total_quantity",
    "product_diversity",
    "category_diversity",
    "score_recency",
    "score_frequency",
    "score_monetary",
    "rfm_score",
    "rfm_normalized_score",
    "delta_recency_latency"
]

categorical_features = [
    "segment",
    "cluster"
]

feature_columns = numeric_features + categorical_features

X = model_data[feature_columns].copy()
y = model_data["churn"].copy()

X.head()

,recency,frequency,monetary,average_latency,observed_intervals,total_quantity,product_diversity,category_diversity,score_recency,score_frequency,score_monetary,rfm_score,rfm_normalized_score,delta_recency_latency,segment,cluster
0,11,2,5741.96,730.0,1,3,3,2,-1.662684,2.719297,0.818697,1.258481,0.868062,-719.0,High Value but Cooling,1
1,13,2,5794.92,732.0,1,7,7,3,-1.622934,2.719297,0.826561,1.241594,0.866566,-719.0,High Value but Cooling,1
2,29,2,5694.98,724.0,1,2,2,1,-1.376703,2.719297,0.811652,1.106469,0.854590,-695.0,High Value but Cooling,1
3,24,2,5718.95,738.0,1,5,5,3,-1.444056,2.719297,0.815255,1.143202,0.857846,-714.0,High Value but Cooling,1
4,7,2,5776.95,730.0,1,4,4,2,-1.754208,2.719297,0.823902,1.308545,0.872499,-723.0,High Value but Cooling,1


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=seed,
    stratify=y
)

split_summary = pd.DataFrame({
    "sample": ["Train", "Test"],
    "customers": [len(y_train), len(y_test)],
    "churn_rate": [y_train.mean(), y_test.mean()]
})

save_table(split_summary, "split-summary")

split_summary

,sample,customers,churn_rate
0,Train,4344,0.010359
1,Test,1862,0.010741


## Model Benchmark

The benchmark compares three model families and three sampling strategies.

The objective is not to find a theoretically perfect churn model, but to identify a practical classifier that separates higher-risk customers from lower-risk customers with enough reliability to support prioritisation.

The models are evaluated using:

- ROC AUC.
- Average Precision.
- Accuracy.
- Balanced Accuracy.
- Precision.
- Recall.
- F1 Score.

In [23]:
def get_sampling_step(strategy):
    """
    Return the resampling object associated with a modelling strategy.

    Inputs:
    -------
        strategy (str): Resampling strategy. Expected values are Original,
            Undersampling, or SMOTE.

    Outputs:
    --------
        object or None: Imbalanced-learn sampler for the selected strategy, or
            None when the original class distribution is preserved.

    Author:
    -------
        Rodrigo Kang
    """

    if strategy == "Original":
        return None
    if strategy == "Undersampling":
        return RandomUnderSampler(random_state=seed)
    if strategy == "SMOTE":
        return SMOTE(random_state=seed, k_neighbors=5)
    raise ValueError(f"Unknown sampling strategy: {strategy}")


def build_pipeline(model, sampling_strategy):
    """
    Build a preprocessing, resampling, and classification pipeline.

    Inputs:
    -------
        model: Scikit-learn compatible classifier with predict_proba support.
        sampling_strategy (str): Resampling strategy used to handle class
            imbalance.

    Outputs:
    --------
        imblearn.pipeline.Pipeline: End-to-end modelling pipeline.

    Author:
    -------
        Rodrigo Kang
    """

    preprocessing = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric_features
            ),
            (
                "categorical",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore"))
                ]),
                categorical_features
            )
        ],
        remainder="drop"
    )

    sampling_step = get_sampling_step(sampling_strategy)

    steps = [("preprocessing", preprocessing)]

    if sampling_step is not None:
        steps.append(("sampling", sampling_step))

    steps.append(("model", model))

    return ImbPipeline(steps=steps)


def evaluate_predictions(y_true, y_probability, threshold=0.50):
    """
    Evaluate probabilistic churn predictions at a fixed classification threshold.

    Inputs:
    -------
        y_true (pd.Series or np.ndarray): Observed binary churn labels.
        y_probability (np.ndarray): Predicted churn probabilities.
        threshold (float): Probability threshold used to convert probabilities
            into class predictions.

    Outputs:
    --------
        dict: ROC AUC, Average Precision, Accuracy, Balanced Accuracy,
            Precision, Recall, and F1 Score.

    Author:
    -------
        Rodrigo Kang
    """

    y_pred = (y_probability >= threshold).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, y_probability),
        "average_precision": average_precision_score(y_true, y_probability),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }

In [24]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=seed,
        class_weight=None
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=10,
        random_state=seed,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=seed,
        n_jobs=-1
    )
}

sampling_strategies = [
    "Original",
    "Undersampling",
    "SMOTE"
]

In [25]:
benchmark_results = []
fitted_models = {}

for model_name, model in models.items():
    for sampling_strategy in sampling_strategies:

        pipeline = build_pipeline(
            model=model,
            sampling_strategy=sampling_strategy
        )

        pipeline.fit(X_train, y_train)

        y_probability = pipeline.predict_proba(X_test)[:, 1]
        metrics = evaluate_predictions(y_test, y_probability)

        row = {
            "model": model_name,
            "sampling_strategy": sampling_strategy,
            **metrics
        }

        benchmark_results.append(row)
        fitted_models[(model_name, sampling_strategy)] = pipeline

benchmark = (
    pd.DataFrame(benchmark_results)
    .sort_values(
        ["roc_auc", "average_precision", "f1"],
        ascending=False
    )
    .reset_index(drop=True)
)

save_table(benchmark.round(4), "model-benchmark")

benchmark.round(4)

,model,sampling_strategy,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1
0,Random Forest,SMOTE,0.9996,0.9707,0.9973,0.9986,0.8000,1.0,0.8889
1,XGBoost,Undersampling,0.9995,0.9506,0.9871,0.9935,0.4545,1.0,0.6250
2,Random Forest,Original,0.9995,0.9585,0.9973,0.9492,0.8571,0.9,0.8780
3,XGBoost,SMOTE,0.9993,0.9604,0.9973,0.9492,0.8571,0.9,0.8780
4,XGBoost,Original,0.9993,0.9511,0.9979,0.9495,0.9000,0.9,0.9000
5,Random Forest,Undersampling,0.9993,0.9387,0.9646,0.9821,0.2326,1.0,0.3774
6,Logistic Regression,SMOTE,0.9986,0.8923,0.9871,0.9935,0.4545,1.0,0.6250
7,Logistic Regression,Original,0.9980,0.8706,0.9957,0.8495,0.8750,0.7,0.7778
8,Logistic Regression,Undersampling,0.9969,0.8292,0.9689,0.9843,0.2564,1.0,0.4082


In [26]:
best_model_name = benchmark.loc[0, "model"]
best_sampling_strategy = benchmark.loc[0, "sampling_strategy"]

best_model = fitted_models[(best_model_name, best_sampling_strategy)]

best_model_name, best_sampling_strategy

('Random Forest', 'SMOTE')

## Best Model Assessment

The best benchmarked model is now evaluated in more detail using the held-out test sample.

The following diagnostics are produced:

- Confusion matrix.
- Classification report.
- ROC curve.
- Precision-Recall curve.
- Kolmogorov-Smirnov statistic and cumulative distribution plot.

In [27]:
y_test_probability = best_model.predict_proba(X_test)[:, 1]
y_test_prediction = (y_test_probability >= 0.50).astype(int)

best_metrics = evaluate_predictions(y_test, y_test_probability)

best_metrics_table = pd.DataFrame([best_metrics]).round(4)
save_table(best_metrics_table, "best-model-metrics")

best_metrics_table

,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1
0,0.9996,0.9707,0.9973,0.9986,0.8,1.0,0.8889


In [28]:
cm = confusion_matrix(y_test, y_test_prediction)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Retention", "Actual Churn"],
    columns=["Predicted Retention", "Predicted Churn"]
)

save_table(cm_df, "confusion-matrix", index=True)

cm_df

,Predicted Retention,Predicted Churn
Actual Retention,1837,5
Actual Churn,0,20


In [29]:
fig = px.imshow(
    cm_df,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels=dict(
        x="Predicted class",
        y="Actual class",
        color="Customers"
    )
)

fig.update_traces(
    textfont_size=16,
    xgap=2,
    ygap=2,
    hovertemplate=(
        "Actual: %{y}<br>"
        "Predicted: %{x}<br>"
        "Customers: %{z}<extra></extra>"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Confusion Matrix",
    xaxis_title="Predicted Class",
    yaxis_title="Actual Class"
)

fig.update_yaxes(autorange="reversed")

save_plotly_figure(fig, "confusion-matrix", width=850, height=700)
fig.show()

In [30]:
classification_report_table = pd.DataFrame(
    classification_report(
        y_test,
        y_test_prediction,
        target_names=["Retention", "Churn"],
        zero_division=0,
        output_dict=True
    )
).T

save_table(classification_report_table.round(4), "classification-report", index=True)

classification_report_table.round(4)

,precision,recall,f1-score,support
Retention,1.0000,0.9973,0.9986,1842.0000
Churn,0.8000,1.0000,0.8889,20.0000
accuracy,0.9973,0.9973,0.9973,0.9973
macro avg,0.9000,0.9986,0.9438,1862.0000
weighted avg,0.9979,0.9973,0.9975,1862.0000


In [31]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_test_probability)
roc_auc = roc_auc_score(y_test, y_test_probability)

roc_curve_table = pd.DataFrame({
    "false_positive_rate": fpr,
    "true_positive_rate": tpr,
    "threshold": roc_thresholds
})

save_table(roc_curve_table, "roc-curve")

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"Model ROC AUC = {roc_auc:.3f}",
        line=dict(width=3),
        hovertemplate=(
            "False positive rate: %{x:.3f}<br>"
            "True positive rate: %{y:.3f}<extra></extra>"
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Random classifier",
        line=dict(width=2, dash="dash"),
        hoverinfo="skip"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Receiver Operating Characteristic Curve",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate"
)

fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1])

save_plotly_figure(fig, "roc-curve")
fig.show()

In [32]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_test_probability)
average_precision = average_precision_score(y_test, y_test_probability)

precision_recall_table = pd.DataFrame({
    "recall": recall,
    "precision": precision,
    "threshold": np.append(pr_thresholds, np.nan)
})

save_table(precision_recall_table, "precision-recall-curve")

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=recall,
        y=precision,
        mode="lines",
        name=f"Average Precision = {average_precision:.3f}",
        line=dict(width=3),
        hovertemplate=(
            "Recall: %{x:.3f}<br>"
            "Precision: %{y:.3f}<extra></extra>"
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[y_test.mean(), y_test.mean()],
        mode="lines",
        name="Baseline churn rate",
        line=dict(width=2, dash="dash"),
        hoverinfo="skip"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Precision-Recall Curve",
    xaxis_title="Recall",
    yaxis_title="Precision"
)

fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1])

save_plotly_figure(fig, "precision-recall-curve")
fig.show()

In [33]:
def ks_table(y_true, y_probability):
    """
    Build the cumulative distributions required for the KS statistic.

    Inputs:
    -------
        y_true (pd.Series or np.ndarray): Observed binary churn labels.
        y_probability (np.ndarray): Predicted churn probabilities.

    Outputs:
    --------
        pd.DataFrame: Ranked predictions with cumulative churn, cumulative
            non-churn, and KS distance columns.

    Author:
    -------
        Rodrigo Kang
    """

    data = pd.DataFrame({
        "actual": np.asarray(y_true),
        "probability": y_probability
    }).sort_values("probability", ascending=False)

    data["non_churn"] = 1 - data["actual"]
    data["churn"] = data["actual"]

    data["cum_churn"] = data["churn"].cumsum() / data["churn"].sum()
    data["cum_non_churn"] = data["non_churn"].cumsum() / data["non_churn"].sum()
    data["ks"] = data["cum_churn"] - data["cum_non_churn"]

    return data

ks_data = ks_table(y_test, y_test_probability)
ks_statistic = ks_data["ks"].max()
ks_probability = ks_data.loc[ks_data["ks"].idxmax(), "probability"]

ks_statistic, ks_probability

save_table(ks_data, "ks-table")

ks_statistic, ks_probability

(0.99728555917481, np.float64(0.5081522689689916))

In [34]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=ks_data["probability"],
        y=ks_data["cum_churn"],
        mode="lines",
        name="Cumulative Churn",
        line=dict(width=3),
        hovertemplate=(
            "Probability: %{x:.3f}<br>"
            "Cumulative churn: %{y:.3f}<extra></extra>"
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=ks_data["probability"],
        y=ks_data["cum_non_churn"],
        mode="lines",
        name="Cumulative Retention",
        line=dict(width=3),
        hovertemplate=(
            "Probability: %{x:.3f}<br>"
            "Cumulative retention: %{y:.3f}<extra></extra>"
        )
    )
)

fig.add_vline(
    x=ks_probability,
    line_dash="dash",
    annotation_text=f"KS = {ks_statistic:.3f}",
    annotation_position="top right"
)

fig = apply_plotly_layout(
    fig,
    title="Kolmogorov-Smirnov Cumulative Curves",
    xaxis_title="Predicted Churn Probability",
    yaxis_title="Cumulative Share"
)

fig.update_xaxes(autorange="reversed")
fig.update_yaxes(range=[0, 1])

save_plotly_figure(fig, "ks-curve")
fig.show()

## Lift and Gains Analysis

ROC and Precision-Recall curves are useful for model assessment, but retention work also needs a business-facing question: how many future churners can be captured if the company targets only the highest-risk customers?

The lift and gains table ranks customers by predicted churn probability and groups them into deciles. This connects the model output to a practical retention campaign, where the team may only have budget to contact the top 5%, 10%, or 20% of at-risk customers.

In [35]:
def build_lift_gains_table(y_true, y_probability, groups=10):
    """
    Build a lift and cumulative gains table from predicted churn probabilities.

    Inputs:
    -------
        y_true (pd.Series or np.ndarray): Observed binary churn labels.
        y_probability (np.ndarray): Predicted churn probabilities.
        groups (int): Number of ranked groups used to summarise the population.

    Outputs:
    --------
        pd.DataFrame: Decile-level table with customers, churners, response rate,
            lift, cumulative gain, and cumulative lift.

    Author:
    -------
        Rodrigo Kang
    """

    gains_data = pd.DataFrame({
        "actual": np.asarray(y_true),
        "probability": y_probability
    }).sort_values("probability", ascending=False).reset_index(drop=True)

    gains_data["rank"] = np.arange(1, len(gains_data) + 1)
    gains_data["decile"] = pd.qcut(
        gains_data["rank"],
        q=groups,
        labels=False,
        duplicates="drop"
    ) + 1

    overall_churn_rate = gains_data["actual"].mean()
    total_churners = gains_data["actual"].sum()
    total_customers = len(gains_data)

    lift_table = (
        gains_data
        .groupby("decile")
        .agg(
            customers=("actual", "size"),
            churners=("actual", "sum"),
            min_probability=("probability", "min"),
            max_probability=("probability", "max")
        )
        .reset_index()
    )

    lift_table["customer_share"] = lift_table["customers"] / total_customers
    lift_table["churn_rate"] = lift_table["churners"] / lift_table["customers"]
    lift_table["lift"] = lift_table["churn_rate"] / overall_churn_rate
    lift_table["cumulative_customers"] = lift_table["customers"].cumsum()
    lift_table["cumulative_churners"] = lift_table["churners"].cumsum()
    lift_table["cumulative_customer_share"] = lift_table["cumulative_customers"] / total_customers
    lift_table["cumulative_gain"] = lift_table["cumulative_churners"] / total_churners
    lift_table["cumulative_lift"] = (
        lift_table["cumulative_gain"]
        / lift_table["cumulative_customer_share"]
    )

    return lift_table

lift_gains_table = build_lift_gains_table(
    y_true=y_test,
    y_probability=y_test_probability,
    groups=10
)


save_table(lift_gains_table, "lift-gains-table")

lift_gains_table

,decile,customers,churners,min_probability,max_probability,customer_share,churn_rate,lift,cumulative_customers,cumulative_churners,cumulative_customer_share,cumulative_gain,cumulative_lift
0,1,187,20,0.002427,0.999668,0.100430,0.106952,9.957219,187,20,0.100430,1.0,9.957219
1,2,186,0,0.000000,0.002427,0.099893,0.000000,0.000000,373,20,0.200322,1.0,4.991957
2,3,186,0,0.000000,0.000000,0.099893,0.000000,0.000000,559,20,0.300215,1.0,3.330948
3,4,186,0,0.000000,0.000000,0.099893,0.000000,0.000000,745,20,0.400107,1.0,2.499329
4,5,186,0,0.000000,0.000000,0.099893,0.000000,0.000000,931,20,0.500000,1.0,2.000000
5,6,186,0,0.000000,0.000000,0.099893,0.000000,0.000000,1117,20,0.599893,1.0,1.666965
6,7,186,0,0.000000,0.000000,0.099893,0.000000,0.000000,1303,20,0.699785,1.0,1.429010
7,8,186,0,0.000000,0.000000,0.099893,0.000000,0.000000,1489,20,0.799678,1.0,1.250504
8,9,186,0,0.000000,0.000000,0.099893,0.000000,0.000000,1675,20,0.899570,1.0,1.111642
9,10,187,0,0.000000,0.000000,0.100430,0.000000,0.000000,1862,20,1.000000,1.0,1.000000


In [36]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=lift_gains_table["cumulative_customer_share"] * 100,
        y=lift_gains_table["cumulative_gain"] * 100,
        mode="lines+markers",
        name="Model",
        line=dict(width=3),
        marker=dict(size=8),
        hovertemplate=(
            "Customers targeted: %{x:.1f}%<br>"
            "Churners captured: %{y:.1f}%<extra></extra>"
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=[0, 100],
        y=[0, 100],
        mode="lines",
        name="Random targeting",
        line=dict(width=2, dash="dash"),
        hoverinfo="skip"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Cumulative Gains Curve",
    xaxis_title="Customers Targeted (%)",
    yaxis_title="Churners Captured (%)"
)

fig.update_xaxes(range=[0, 100])
fig.update_yaxes(range=[0, 100])

save_plotly_figure(fig, "cumulative-gains-curve")
fig.show()

In [37]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=lift_gains_table["decile"].astype(str),
        y=lift_gains_table["lift"],
        text=lift_gains_table["lift"].round(2),
        textposition="outside",
        name="Lift",
        hovertemplate=(
            "Risk decile: %{x}<br>"
            "Lift: %{y:.2f}<extra></extra>"
        )
    )
)

fig.add_hline(
    y=1,
    line_dash="dash",
    annotation_text="Random targeting",
    annotation_position="top right"
)

fig = apply_plotly_layout(
    fig,
    title="Lift by Risk Decile",
    xaxis_title="Risk Decile",
    yaxis_title="Lift over Random Targeting"
)

fig.update_yaxes(range=[0, max(1.1, lift_gains_table["lift"].max() * 1.15)])

save_plotly_figure(fig, "lift-by-risk-decile")
fig.show()

The lift and gains view translates the model into a retention workflow. Instead of contacting all customers, the business can rank customers by predicted churn probability and focus on the highest-risk deciles. This is often more useful operationally than a single AUC value because it links model performance to campaign capacity.

## Robustness Diagnostics: Importance and Ablation

The benchmark above may look unusually strong because churn in a non-contractual setting is closely related to customer inactivity. That is not necessarily a modelling error: in retail, recency is often the most informative behavioural signal.

However, for a portfolio project it is worth making this dependence visible. The next diagnostics answer two practical questions:

- Which variables are driving the selected model?
- How much performance remains if `recency` and related inactivity variables are removed?

This helps separate genuine temporal validation from a model that is mainly recovering the churn rule embedded in the RFM construction.

In [38]:
def get_model_feature_names(pipeline):
    """
    Extract transformed feature names from a fitted modelling pipeline.

    Inputs:
    -------
        pipeline (imblearn.pipeline.Pipeline): Fitted preprocessing and model
            pipeline.

    Outputs:
    --------
        list: Names of the numerical and one-hot encoded categorical features
            after preprocessing.

    Author:
    -------
        Rodrigo Kang
    """

    preprocessing = pipeline.named_steps["preprocessing"]
    feature_names = preprocessing.get_feature_names_out()

    clean_feature_names = [
        name
        .replace("numeric__", "")
        .replace("categorical__", "")
        for name in feature_names
    ]

    return clean_feature_names


def extract_model_importance(pipeline):
    """
    Extract model-driven feature importance from the selected classifier.

    Inputs:
    -------
        pipeline (imblearn.pipeline.Pipeline): Fitted preprocessing, optional
            resampling, and classifier pipeline.

    Outputs:
    --------
        pd.DataFrame: Feature importance table sorted from highest to lowest
            importance. Tree models use feature_importances_; logistic
            regression uses absolute standardized coefficients.

    Author:
    -------
        Rodrigo Kang
    """

    model = pipeline.named_steps["model"]
    feature_names = get_model_feature_names(pipeline)

    if hasattr(model, "feature_importances_"):
        importance_values = model.feature_importances_
        importance_type = "model_feature_importance"
    elif hasattr(model, "coef_"):
        importance_values = np.abs(model.coef_).ravel()
        importance_type = "absolute_coefficient"
    else:
        raise ValueError("The selected model does not expose feature importances or coefficients.")

    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": importance_values,
        "importance_type": importance_type
    })

    importance["importance_share"] = (
        importance["importance"] / importance["importance"].sum()
    )

    return (
        importance
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

feature_importance = extract_model_importance(best_model)

save_table(feature_importance.round(6), "feature-importance")

feature_importance.head(20).round(4)

,feature,importance,importance_type,importance_share
0,delta_recency_latency,0.2035,model_feature_importance,0.2035
1,cluster_3,0.1281,model_feature_importance,0.1281
2,segment_Inactive Low-Value Customers,0.1145,model_feature_importance,0.1145
3,score_monetary,0.0935,model_feature_importance,0.0935
4,monetary,0.0912,model_feature_importance,0.0912
5,score_frequency,0.0808,model_feature_importance,0.0808
6,total_quantity,0.0562,model_feature_importance,0.0562
7,observed_intervals,0.0500,model_feature_importance,0.0500
8,frequency,0.0419,model_feature_importance,0.0419
9,product_diversity,0.0353,model_feature_importance,0.0353


In [39]:
top_features = feature_importance.head(15).sort_values("importance")

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=top_features["importance"],
        y=top_features["feature"],
        orientation="h",
        text=top_features["importance"].round(4),
        textposition="outside",
        name="Importance",
        hovertemplate=(
            "Feature: %{y}<br>"
            "Importance: %{x:.4f}<extra></extra>"
        )
    )
)

fig = apply_plotly_layout(
    fig,
    title="Top Feature Importances for the Selected Churn Model",
    xaxis_title="Importance",
    yaxis_title=None
)

fig.update_layout(showlegend=False)
fig.update_xaxes(range=[0, top_features["importance"].max() * 1.20])

save_plotly_figure(fig, "top-feature-importances", width=1050, height=750)
fig.show()

In [40]:
def build_pipeline_for_features(model, sampling_strategy, numeric_cols, categorical_cols):
    """
    Build a modelling pipeline for a custom feature subset.

    Inputs:
    -------
        model: Scikit-learn compatible classifier with predict_proba support.
        sampling_strategy (str): Resampling strategy used to handle class
            imbalance.
        numeric_cols (list): Numerical feature names included in the pipeline.
        categorical_cols (list): Categorical feature names included in the
            pipeline.

    Outputs:
    --------
        imblearn.pipeline.Pipeline: End-to-end modelling pipeline for the
            requested feature subset.

    Author:
    -------
        Rodrigo Kang
    """

    preprocessing = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric_cols
            ),
            (
                "categorical",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore"))
                ]),
                categorical_cols
            )
        ],
        remainder="drop"
    )

    sampling_step = get_sampling_step(sampling_strategy)
    steps = [("preprocessing", preprocessing)]

    if sampling_step is not None:
        steps.append(("sampling", sampling_step))

    steps.append(("model", model))

    return ImbPipeline(steps=steps)


def run_ablation_study(model_name, sampling_strategy, feature_sets):
    """
    Refit the selected model across alternative feature sets.

    Inputs:
    -------
        model_name (str): Name of the model family selected from the benchmark.
        sampling_strategy (str): Resampling strategy selected from the benchmark.
        feature_sets (dict): Mapping where each key is a scenario name and each
            value contains numeric and categorical feature lists.

    Outputs:
    --------
        pd.DataFrame: Performance metrics for each feature set evaluated on the
            same held-out test sample.

    Author:
    -------
        Rodrigo Kang
    """

    ablation_results = []

    for feature_set_name, feature_set in feature_sets.items():
        scenario_numeric = feature_set.get("numeric", [])
        scenario_categorical = feature_set.get("categorical", [])
        scenario_features = scenario_numeric + scenario_categorical

        scenario_model = clone(models[model_name])
        scenario_pipeline = build_pipeline_for_features(
            model=scenario_model,
            sampling_strategy=sampling_strategy,
            numeric_cols=scenario_numeric,
            categorical_cols=scenario_categorical
        )

        scenario_pipeline.fit(
            X_train[scenario_features],
            y_train
        )

        scenario_probability = scenario_pipeline.predict_proba(
            X_test[scenario_features]
        )[:, 1]

        scenario_metrics = evaluate_predictions(
            y_test,
            scenario_probability
        )

        ablation_results.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "sampling_strategy": sampling_strategy,
            "n_features": len(scenario_features),
            **scenario_metrics
        })

    return (
        pd.DataFrame(ablation_results)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

ablation_feature_sets = {
    "Recency only": {
        "numeric": ["recency"],
        "categorical": []
    },
    "Frequency + Monetary": {
        "numeric": ["frequency", "monetary"],
        "categorical": []
    },
    "RFM raw": {
        "numeric": ["recency", "frequency", "monetary"],
        "categorical": []
    },
    "RFM scores": {
        "numeric": [
            "score_recency",
            "score_frequency",
            "score_monetary",
            "rfm_normalized_score"
        ],
        "categorical": []
    },
    "RFM raw + Segment": {
        "numeric": ["recency", "frequency", "monetary"],
        "categorical": ["segment", "cluster"]
    },
    "No recency family": {
        "numeric": [
            "frequency",
            "monetary",
            "observed_intervals",
            "total_quantity",
            "product_diversity",
            "category_diversity",
            "score_frequency",
            "score_monetary"
        ],
        "categorical": ["segment", "cluster"]
    },
    "Full model": {
        "numeric": numeric_features,
        "categorical": categorical_features
    }
}

ablation_results = run_ablation_study(
    model_name=best_model_name,
    sampling_strategy=best_sampling_strategy,
    feature_sets=ablation_feature_sets
)


save_table(ablation_results.round(4), "ablation-results")

ablation_results.round(4)

,feature_set,model,sampling_strategy,n_features,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1
0,Full model,Random Forest,SMOTE,16,0.9996,0.9707,0.9973,0.9986,0.8000,1.00,0.8889
1,RFM scores,Random Forest,SMOTE,4,0.9995,0.9528,0.9979,0.9989,0.8333,1.00,0.9091
2,RFM raw,Random Forest,SMOTE,3,0.9993,0.9478,0.9968,0.9984,0.7692,1.00,0.8696
3,RFM raw + Segment,Random Forest,SMOTE,5,0.9993,0.9449,0.9968,0.9984,0.7692,1.00,0.8696
4,No recency family,Random Forest,SMOTE,10,0.9948,0.6438,0.9860,0.9682,0.4318,0.95,0.5938
5,Frequency + Monetary,Random Forest,SMOTE,2,0.9926,0.4654,0.9855,0.8938,0.4103,0.80,0.5424
6,Recency only,Random Forest,SMOTE,1,0.9666,0.4362,0.9834,0.9421,0.3830,0.90,0.5373


In [41]:
plot_data = ablation_results.sort_values("roc_auc")

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=plot_data["roc_auc"],
        y=plot_data["feature_set"],
        orientation="h",
        text=plot_data["roc_auc"].round(3),
        textposition="outside",
        name="ROC AUC",
        hovertemplate=(
            "Feature set: %{y}<br>"
            "ROC AUC: %{x:.4f}<extra></extra>"
        )
    )
)

full_model_auc = (
    ablation_results
    .loc[ablation_results["feature_set"] == "Full model", "roc_auc"]
    .iloc[0]
)

fig.add_vline(
    x=full_model_auc,
    line_dash="dash",
    annotation_text="Full model",
    annotation_position="top"
)

fig = apply_plotly_layout(
    fig,
    title="Ablation Study: ROC AUC by Feature Set",
    xaxis_title="ROC AUC",
    yaxis_title=None
)

fig.update_layout(showlegend=False)
fig.update_xaxes(range=[0, 1.05])

save_plotly_figure(fig, "ablation-study-roc-auc", width=1050, height=700)
fig.show()

### Robustness Interpretation

The ablation study indicates that customer inactivity is a major driver of predictive performance. The feature importance analysis identifies `delta_recency_latency` as the most influential predictor, while inactivity-related segments such as *Inactive Low-Value Customers* also rank among the most important variables.

However, recency alone does not fully explain the observed churn patterns. Although recency-based features contribute substantially to predictive performance, the strongest results are obtained when they are combined with information about purchase frequency, monetary value, and customer segmentation.

This conclusion is supported by the ablation analysis. Removing the recency family leads to a noticeable deterioration in Average Precision and F1 score, indicating that inactivity contains valuable information about future customer behaviour. At the same time, the *Recency Only* model performs worse than the richer feature sets, suggesting that additional behavioural information provides meaningful incremental value.

From a business perspective, the results indicate that customers become increasingly likely to churn when their time since last purchase exceeds their typical purchasing rhythm. Customer value, purchasing intensity, and segment membership provide additional context that helps distinguish temporary inactivity from more persistent disengagement.

## Model Summary Table

The final diagnostic summary consolidates the main out-of-sample classification metrics. This table is intended as a compact checkpoint before scoring the full customer base.

In [42]:
model_summary_table = pd.DataFrame({
    "metric": [
        "ROC AUC",
        "Average Precision",
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "KS Statistic",
        "Decision Threshold"
    ],
    "value": [
        roc_auc_score(y_test, y_test_probability),
        average_precision_score(y_test, y_test_probability),
        accuracy_score(y_test, y_test_prediction),
        balanced_accuracy_score(y_test, y_test_prediction),
        precision_score(y_test, y_test_prediction, zero_division=0),
        recall_score(y_test, y_test_prediction, zero_division=0),
        f1_score(y_test, y_test_prediction, zero_division=0),
        ks_statistic,
        0.50
    ]
})

model_summary_table["value"] = model_summary_table["value"].round(4)

save_table(model_summary_table, "model-summary")

model_summary_table

,metric,value
0,ROC AUC,0.9996
1,Average Precision,0.9707
2,Accuracy,0.9973
3,Balanced Accuracy,0.9986
4,Precision,0.8000
5,Recall,1.0000
6,F1 Score,0.8889
7,KS Statistic,0.9973
8,Decision Threshold,0.5000


## Score Customers

The final step applies the selected model to the full calibration customer base. The output is a customer-level table containing RFM characteristics, segment labels, observed validation churn, predicted churn probability, and predicted churn class.

This dataset can be used to prioritise retention actions or to feed the CLV workflow developed later in the project.

In [43]:
model_data["churn_probability"] = best_model.predict_proba(X)[:, 1]
model_data["churn_prediction"] = (model_data["churn_probability"] >= 0.50).astype(int)

final_churn_output = (
    model_data[
        [
            "customer_id",
            "recency",
            "frequency",
            "monetary",
            "average_latency",
            "cluster",
            "segment",
            "rfm_normalized_score",
            "rfm_churn_rule",
            "returned_in_validation",
            "churn",
            "churn_probability",
            "churn_prediction"
        ]
    ]
    .sort_values("churn_probability", ascending=False)
    .reset_index(drop=True)
)

final_churn_output.head(20)

,customer_id,recency,frequency,monetary,average_latency,cluster,segment,rfm_normalized_score,rfm_churn_rule,returned_in_validation,churn,churn_probability,churn_prediction
0,29906,458,4,87491.355000,91.000000,3,Inactive Low-Value Customers,0.775080,1,0,1,0.999668,1
1,29813,458,4,48761.856000,91.000000,3,Inactive Low-Value Customers,0.766098,1,0,1,0.999661,1
2,29582,458,4,153310.578700,91.000000,3,Inactive Low-Value Customers,0.783043,1,0,1,0.999652,1
3,29749,458,4,137715.853800,91.000000,3,Inactive Low-Value Customers,0.781568,1,0,1,0.999652,1
4,29742,458,4,138840.843600,91.000000,3,Inactive Low-Value Customers,0.781680,1,0,1,0.999652,1
5,29773,458,4,85177.081200,91.000000,3,Inactive Low-Value Customers,0.774684,1,0,1,0.999652,1
6,29774,458,4,175502.157400,91.000000,3,Inactive Low-Value Customers,0.784872,1,0,1,0.999652,1
7,29510,458,4,125357.039456,91.000000,3,Inactive Low-Value Customers,0.780256,1,0,1,0.999652,1
8,29748,427,4,131600.935400,91.000000,3,Inactive Low-Value Customers,0.787191,1,0,1,0.999626,1
9,29921,427,4,193258.670800,91.000000,3,Inactive Low-Value Customers,0.792409,1,0,1,0.999611,1


In [44]:
output_path = (
    Path("../data/processed/python")
    / "customer-churn-prediction.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

final_churn_output.to_csv(
    output_path,
    index=False
)

output_path

WindowsPath('../data/processed/python/customer-churn-prediction.csv')

## Final Remarks

This notebook extends the RFM segmentation workflow into a supervised churn modelling framework for non-contractual retail relationships.

The key modelling decision is the temporal split between calibration and validation. RFM features are computed only from historical behaviour, while churn is assessed using the following year of transactions. This produces a more realistic setup than defining churn from the full observation window.

The benchmark compares Logistic Regression, Random Forest, and XGBoost under different sampling strategies. The selected model is evaluated using classification metrics, ROC and Precision-Recall curves, the Kolmogorov-Smirnov statistic, and lift/gains analysis.

The lift and gains section connects the model to a practical retention campaign: customers can be ranked by predicted churn probability and targeted according to available business capacity.

Feature importance and ablation analyses are included to better understand the drivers of predictive performance. The results indicate that customer inactivity is a dominant signal, particularly when recency is evaluated relative to each customer's historical purchasing rhythm. At the same time, frequency, monetary value, and customer segmentation provide additional information that improves performance beyond recency alone.

The resulting customer-level output provides churn probabilities that can be used for retention prioritisation and later combined with customer value estimates in the CLV section of the project.